# AgentCore Gateway Demo — Infrastructure Setup

This notebook sets up the infrastructure for demonstrating **centralized MCP server management** with **auth propagation** using AWS AgentCore Gateway.

## Architecture

```
 User (CLI)
   │
   │ username + password (Resource Owner Password grant)
   ▼
 Okta Token Endpoint ──▶ JWT (with sub, groups claims)
   │
   ▼
 Strands Agent (JWT attached as Bearer token)
   │
   │ MCP (StreamableHTTP + Bearer JWT)
   ▼
 AgentCore Gateway
   ├── Ingress Auth: validates JWT (signature, audience, client_id)
   ├── JWT claims mapped to Cedar principal tags:
   │     groups claim ──▶ principal tag "groups"
   ├── Cedar Policy Engine (ENFORCE mode)
   │     • principal is AgentCore::OAuthUser
   │     • action == AgentCore::Action::"<TargetName>"
   │     • resource == AgentCore::Gateway::"<gateway-arn>"
   │     • condition: principal.hasTag("groups")
   │       && principal.getTag("groups") like "*finance-admins*"
   │
   ├───────────────┬───────────────┐
   ▼               ▼               ▼
 Lambda         Lambda          MCP Server
 (Weather)      (Finance)       (ECS)
```

## What this notebook does

1. Install dependencies and load Okta config
2. Deploy **all** infrastructure via Terraform (ECR, ECS, Lambda, IAM, networking)
   - **2b. (OPTIONAL)** Discover existing infrastructure via AWS APIs — skip Terraform if already deployed
3. Build and push Docker image + wait for ECS task
4. Verify Lambda functions
5. Create AgentCore Gateway with Okta OIDC auth
6. Define Cedar policies for group-based access control (ENFORCE mode)
7. Save config for the demo notebook

## Cell 1: Dependencies + Configuration

Before running this notebook:
1. Create an Okta Developer account at https://developer.okta.com
2. Create an OIDC Web Application
3. Create groups: `analysts` and `finance-admins`
4. Create test users: Alice (analysts group) and Bob (finance-admins group)
5. Configure Okta to include `groups` claim in tokens
6. Copy `.env.example` to `.env` and fill in your values

In [ ]:
%pip install -q boto3 strands-agents strands-agents-tools mcp httpx python-dotenv PyJWT requests

import os
import json
import time
import boto3
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Okta Configuration ---
OKTA_DOMAIN = os.environ["OKTA_DOMAIN"]           # e.g., "dev-12345678.okta.com"
OKTA_CLIENT_ID = os.environ["OKTA_CLIENT_ID"]     # OIDC app client ID
OKTA_CLIENT_SECRET = os.environ["OKTA_CLIENT_SECRET"]
OKTA_ISSUER = f"https://{OKTA_DOMAIN}/oauth2/default"
OKTA_JWKS_URI = f"{OKTA_ISSUER}/v1/keys"

# --- AWS Configuration ---
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts", region_name=AWS_REGION).get_caller_identity()["Account"]

# --- Demo naming ---
DEMO_PREFIX = "agentcore-mcp-demo"

print(f"Okta Domain:    {OKTA_DOMAIN}")
print(f"Okta Issuer:    {OKTA_ISSUER}")
print(f"AWS Region:     {AWS_REGION}")
print(f"AWS Account:    {ACCOUNT_ID}")
print(f"Demo Prefix:    {DEMO_PREFIX}")
print("\n✅ Configuration loaded successfully")

## Cell 2: Deploy All Infrastructure (Terraform)

This runs `terraform apply` to create **all** AWS infrastructure:
- **ECR** repository for the CRM MCP server Docker image
- **ECS** cluster, task definition, Fargate service, security group
- **Lambda** functions (Weather + Finance) with inline Python code
- **IAM** roles for both ECS and Lambda execution

After this cell, we build and push the Docker image (Cell 3), then verify Lambdas (Cell 4).

In [ ]:
import subprocess

TERRAFORM_DIR = os.path.join(os.getcwd(), "terraform")

# --- Step 1: Terraform init ---
print("Initializing Terraform...")
subprocess.run(["terraform", "init", "-upgrade"], cwd=TERRAFORM_DIR, check=True)

# --- Step 2: Apply ECR repo first (needed for docker push) ---
print("\nCreating ECR repository via Terraform...")
subprocess.run(
    ["terraform", "apply", "-auto-approve", "-target=aws_ecr_repository.crm_mcp",
     f"-var=aws_region={AWS_REGION}", f"-var=demo_prefix={DEMO_PREFIX}",
     "-var=container_image=placeholder"],
    cwd=TERRAFORM_DIR, check=True,
)

# Get ECR URI from Terraform output
result = subprocess.run(
    ["terraform", "output", "-raw", "ecr_repository_url"],
    cwd=TERRAFORM_DIR, capture_output=True, text=True, check=True,
)
ecr_uri = result.stdout.strip()
image_tag = f"{ecr_uri}:latest"
print(f"ECR repo: {ecr_uri}")

# --- Step 3: Full Terraform apply (ECS, Lambda, IAM, networking) ---
print("\nApplying full Terraform configuration...")
subprocess.run(
    ["terraform", "apply", "-auto-approve",
     f"-var=aws_region={AWS_REGION}", f"-var=demo_prefix={DEMO_PREFIX}",
     f"-var=container_image={image_tag}"],
    cwd=TERRAFORM_DIR, check=True,
)

# --- Step 4: Capture all Terraform outputs ---
tf_outputs = {}
for key in ["ecs_cluster_name", "ecs_service_name", "security_group_id",
            "weather_lambda_arn", "weather_lambda_name",
            "finance_lambda_arn", "finance_lambda_name", "lambda_role_name",
            "gateway_role_arn"]:
    r = subprocess.run(
        ["terraform", "output", "-raw", key],
        cwd=TERRAFORM_DIR, capture_output=True, text=True, check=True,
    )
    tf_outputs[key] = r.stdout.strip()

ECS_CLUSTER_NAME = tf_outputs["ecs_cluster_name"]
ECS_SERVICE_NAME = tf_outputs["ecs_service_name"]
ECR_REPO_NAME = f"{DEMO_PREFIX}-crm-mcp"
sg_id = tf_outputs["security_group_id"]
WEATHER_FUNCTION_NAME = tf_outputs["weather_lambda_name"]
FINANCE_FUNCTION_NAME = tf_outputs["finance_lambda_name"]
WEATHER_LAMBDA_ARN = tf_outputs["weather_lambda_arn"]
FINANCE_LAMBDA_ARN = tf_outputs["finance_lambda_arn"]
LAMBDA_ROLE_NAME = tf_outputs["lambda_role_name"]
GATEWAY_ROLE_ARN = tf_outputs["gateway_role_arn"]

print(f"\n--- Terraform Outputs ---")
print(f"ECS cluster:     {ECS_CLUSTER_NAME}")
print(f"ECS service:     {ECS_SERVICE_NAME}")
print(f"Weather Lambda:  {WEATHER_FUNCTION_NAME}")
print(f"Finance Lambda:  {FINANCE_FUNCTION_NAME}")
print(f"Lambda role:     {LAMBDA_ROLE_NAME}")
print(f"Gateway role:    {GATEWAY_ROLE_ARN}")
print(f"\n✅ All infrastructure created via Terraform")

## Cell 2b: Discover Existing Infrastructure (OPTIONAL)

**Skip this cell if you ran Cell 2 (Terraform) in this session.**

If the Terraform infrastructure was deployed previously (e.g., in an earlier session), run this cell
to discover all resource identifiers via AWS API calls instead of Terraform outputs. This sets the
same variables that Cell 2 produces, so you can proceed directly to Cell 3 (or Cell 5 if Docker/ECS
is already running).

All resources are discovered using the `DEMO_PREFIX` naming convention (`agentcore-mcp-demo-*`).

In [ ]:
# --- OPTIONAL: Discover existing infrastructure via AWS APIs ---
# Run this instead of Cell 2 if Terraform was deployed in a previous session.

import subprocess

iam_client = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=AWS_REGION)
ecs_client = boto3.client("ecs", region_name=AWS_REGION)
ec2_client = boto3.client("ec2", region_name=AWS_REGION)
ecr_client = boto3.client("ecr", region_name=AWS_REGION)

errors = []

# --- ECS Cluster ---
ECS_CLUSTER_NAME = f"{DEMO_PREFIX}-cluster"
try:
    resp = ecs_client.describe_clusters(clusters=[ECS_CLUSTER_NAME])
    active = [c for c in resp["clusters"] if c["status"] == "ACTIVE"]
    assert active, f"Cluster '{ECS_CLUSTER_NAME}' not found or not ACTIVE"
    print(f"ECS cluster:       {ECS_CLUSTER_NAME}")
except Exception as e:
    errors.append(f"ECS cluster: {e}")

# --- ECS Service ---
ECS_SERVICE_NAME = f"{DEMO_PREFIX}-crm-service"
try:
    resp = ecs_client.describe_services(cluster=ECS_CLUSTER_NAME, services=[ECS_SERVICE_NAME])
    active = [s for s in resp["services"] if s["status"] == "ACTIVE"]
    assert active, f"Service '{ECS_SERVICE_NAME}' not found or not ACTIVE"
    print(f"ECS service:       {ECS_SERVICE_NAME}")
except Exception as e:
    errors.append(f"ECS service: {e}")

# --- ECR Repository ---
ECR_REPO_NAME = f"{DEMO_PREFIX}-crm-mcp"
try:
    resp = ecr_client.describe_repositories(repositoryNames=[ECR_REPO_NAME])
    ecr_uri = resp["repositories"][0]["repositoryUri"]
    image_tag = f"{ecr_uri}:latest"
    print(f"ECR repo:          {ECR_REPO_NAME} ({ecr_uri})")
except Exception as e:
    errors.append(f"ECR repo: {e}")

# --- Security Group ---
sg_id = None
try:
    resp = ec2_client.describe_security_groups(
        Filters=[{"Name": "group-name", "Values": [f"{DEMO_PREFIX}-mcp-sg"]}]
    )
    assert resp["SecurityGroups"], f"Security group '{DEMO_PREFIX}-mcp-sg' not found"
    sg_id = resp["SecurityGroups"][0]["GroupId"]
    print(f"Security group:    {sg_id} ({DEMO_PREFIX}-mcp-sg)")
except Exception as e:
    errors.append(f"Security group: {e}")

# --- Weather Lambda ---
WEATHER_FUNCTION_NAME = f"{DEMO_PREFIX}-weather"
WEATHER_LAMBDA_ARN = None
try:
    resp = lambda_client.get_function(FunctionName=WEATHER_FUNCTION_NAME)
    WEATHER_LAMBDA_ARN = resp["Configuration"]["FunctionArn"]
    print(f"Weather Lambda:    {WEATHER_FUNCTION_NAME} ({WEATHER_LAMBDA_ARN})")
except Exception as e:
    errors.append(f"Weather Lambda: {e}")

# --- Finance Lambda ---
FINANCE_FUNCTION_NAME = f"{DEMO_PREFIX}-finance"
FINANCE_LAMBDA_ARN = None
try:
    resp = lambda_client.get_function(FunctionName=FINANCE_FUNCTION_NAME)
    FINANCE_LAMBDA_ARN = resp["Configuration"]["FunctionArn"]
    print(f"Finance Lambda:    {FINANCE_FUNCTION_NAME} ({FINANCE_LAMBDA_ARN})")
except Exception as e:
    errors.append(f"Finance Lambda: {e}")

# --- Lambda Execution Role ---
LAMBDA_ROLE_NAME = f"{DEMO_PREFIX}-lambda-role"
try:
    resp = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)
    print(f"Lambda role:       {LAMBDA_ROLE_NAME}")
except Exception as e:
    errors.append(f"Lambda role: {e}")

# --- Gateway IAM Role ---
GATEWAY_ROLE_ARN = None
gateway_role_name = f"{DEMO_PREFIX}-gateway-role"
try:
    resp = iam_client.get_role(RoleName=gateway_role_name)
    GATEWAY_ROLE_ARN = resp["Role"]["Arn"]
    print(f"Gateway role:      {GATEWAY_ROLE_ARN}")
except Exception as e:
    errors.append(f"Gateway role: {e}")

# --- Discover ECS MCP Server URL ---
MCP_SERVER_URL = None
try:
    tasks = ecs_client.list_tasks(cluster=ECS_CLUSTER_NAME, serviceName=ECS_SERVICE_NAME)
    if tasks["taskArns"]:
        task_detail = ecs_client.describe_tasks(cluster=ECS_CLUSTER_NAME, tasks=tasks["taskArns"])
        task = task_detail["tasks"][0]
        if task["lastStatus"] == "RUNNING":
            for attachment in task.get("attachments", []):
                for detail in attachment.get("details", []):
                    if detail["name"] == "networkInterfaceId":
                        eni = ec2_client.describe_network_interfaces(
                            NetworkInterfaceIds=[detail["value"]]
                        )
                        public_ip = eni["NetworkInterfaces"][0].get("Association", {}).get("PublicIp")
                        if public_ip:
                            MCP_SERVER_URL = f"http://{public_ip}:8080"
    if MCP_SERVER_URL:
        print(f"MCP Server URL:    {MCP_SERVER_URL}")
    else:
        print(f"MCP Server URL:    (not running — run Cell 3 to deploy)")
except Exception as e:
    print(f"MCP Server URL:    (could not discover: {e})")

# --- Summary ---
if errors:
    print(f"\n--- Missing Resources ({len(errors)}) ---")
    for err in errors:
        print(f"  - {err}")
    print("\nSome resources are missing. Run Cell 2 (Terraform) to create them.")
else:
    print(f"\n All infrastructure discovered via AWS APIs")
    print(f"  You can skip Cell 2 (Terraform) and proceed to Cell 3 or Cell 5.")

## Cell 3: Docker Build/Push + Wait for ECS

Builds the CRM MCP server Docker image, pushes it to ECR, and waits for the ECS Fargate task to get a public IP.

In [ ]:
ecs_client = boto3.client("ecs", region_name=AWS_REGION)
ec2_client = boto3.client("ec2", region_name=AWS_REGION)

ECS_MCP_SERVER_DIR = os.path.join(os.getcwd(), "ecs_mcp_server")

# --- Build and push Docker image (linux/amd64 for Fargate) ---
print("Building and pushing Docker image (linux/amd64)...")
registry = ecr_uri.split("/")[0]

subprocess.run(
    f"aws ecr get-login-password --region {AWS_REGION} | docker login --username AWS --password-stdin {registry}",
    shell=True, check=True, capture_output=True,
)
subprocess.run(["docker", "build", "--platform", "linux/amd64", "-t", image_tag, ECS_MCP_SERVER_DIR], check=True)
subprocess.run(["docker", "push", image_tag], check=True)
print(f"Pushed image: {image_tag}")

# --- Wait for task to be running and get public IP ---
print("\nWaiting for ECS task to start (this may take 1-2 minutes)...")
MCP_SERVER_URL = None
for attempt in range(24):
    tasks = ecs_client.list_tasks(cluster=ECS_CLUSTER_NAME, serviceName=ECS_SERVICE_NAME)
    if tasks["taskArns"]:
        task_detail = ecs_client.describe_tasks(cluster=ECS_CLUSTER_NAME, tasks=tasks["taskArns"])
        task = task_detail["tasks"][0]
        if task["lastStatus"] == "RUNNING":
            eni_id = None
            for attachment in task.get("attachments", []):
                for detail in attachment.get("details", []):
                    if detail["name"] == "networkInterfaceId":
                        eni_id = detail["value"]
            if eni_id:
                eni = ec2_client.describe_network_interfaces(NetworkInterfaceIds=[eni_id])
                MCP_SERVER_URL = f"http://{eni['NetworkInterfaces'][0]['Association']['PublicIp']}:8080"
                print(f"\n✅ MCP Server running at: {MCP_SERVER_URL}")
                break
    time.sleep(5)
    print(f"  Waiting... ({(attempt + 1) * 5}s)")
else:
    print("⚠️  ECS task did not start within expected time. Check ECS console.")

# --- Verification: health check ---
if MCP_SERVER_URL:
    import requests
    try:
        health = requests.get(f"{MCP_SERVER_URL}/health", timeout=10)
        print(f"✅ Health check: {health.json()}")
    except Exception as e:
        print(f"⚠️  Health check failed (may need a moment): {e}")

## Cell 4: Verify Lambda Functions

Invokes both Lambda functions directly to confirm they were deployed correctly by Terraform.

In [ ]:
lambda_client = boto3.client("lambda", region_name=AWS_REGION)

# --- Verify Weather Lambda ---
response = lambda_client.invoke(
    FunctionName=WEATHER_FUNCTION_NAME,
    Payload=json.dumps({"city": "Sydney"})
)
result = json.loads(response["Payload"].read())
print(f"✅ Weather Lambda test: {json.dumps(json.loads(result['body']), indent=2)}")

# --- Verify Finance Lambda ---
response = lambda_client.invoke(
    FunctionName=FINANCE_FUNCTION_NAME,
    Payload=json.dumps({"query": "quarterly revenue"})
)
result = json.loads(response["Payload"].read())
print(f"\n✅ Finance Lambda test: {json.dumps(json.loads(result['body']), indent=2)}")

## Cell 5: Create AgentCore Gateway with Okta OIDC Auth

This creates the AgentCore Gateway with:
- **Okta as the direct OIDC provider** (no Cognito intermediary)
- **3 targets**: Weather Lambda, Finance Lambda, CRM MCP Server on ECS
- **Ingress auth**: Gateway validates Okta JWTs against the JWKS endpoint

**Why `roleArn`?** The Gateway is a managed AWS service that acts on your behalf — like an ECS task or Lambda function, it needs an IAM execution role. The `roleArn` parameter assigns the Gateway an IAM role so it can:
1. **Invoke Lambda targets** — when users call tools like `get_weather`, the Gateway makes the `lambda:InvokeFunction` call using this role
2. **Evaluate Cedar policies** — the policy engine attachment (Cell 6) requires `bedrock-agentcore:*` permissions on this role
3. **Access MCP server targets** — for reaching backend endpoints on your behalf

Without `roleArn`, the Gateway has no AWS identity and cannot reach any of your backends. The role is created by Terraform in Cell 2 (see `gateway_role_arn` output) with a trust policy allowing the AgentCore service to assume it.

In [ ]:
# --- AgentCore Gateway control plane client ---
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

GATEWAY_NAME = f"{DEMO_PREFIX}-gateway"
OKTA_DISCOVERY_URL = f"{OKTA_ISSUER}/.well-known/openid-configuration"

# --- Step 0: Ensure Okta auth server has a 'client_id' claim ---
# AgentCore Gateway checks allowedClients against the 'client_id' claim in the JWT.
# Okta puts the client ID in the 'cid' claim by default, so we add a custom 'client_id'
# claim that maps to the app's client ID.
print("Configuring Okta auth server 'client_id' claim...")
import requests as http_requests
OKTA_API_TOKEN = os.environ.get("OKTA_API_TOKEN", "")
if OKTA_API_TOKEN:
    resp = http_requests.post(
        f"{OKTA_ISSUER.replace('/oauth2/default', '')}/api/v1/authorizationServers/default/claims",
        headers={
            "Authorization": f"SSWS {OKTA_API_TOKEN}",
            "Content-Type": "application/json",
        },
        json={
            "name": "client_id",
            "status": "ACTIVE",
            "claimType": "RESOURCE",
            "valueType": "EXPRESSION",
            "value": "app.clientId",
            "alwaysIncludeInToken": True,
            "conditions": {"scopes": []},
        },
    )
    if resp.status_code == 201:
        print("  Created 'client_id' claim on Okta auth server")
    elif resp.status_code == 409:
        print("  'client_id' claim already exists")
    else:
        print(f"  Warning: Could not create claim ({resp.status_code}): {resp.text[:200]}")
else:
    print("  Skipping (no OKTA_API_TOKEN set) — ensure 'client_id' claim exists manually")

# --- Step 1: Create the Gateway with Okta CUSTOM_JWT authorizer ---
# allowedAudience uses "api://default" — Okta's default authorization server audience.
print("\nCreating AgentCore Gateway with Okta OIDC authorizer...")

try:
    gateway_response = agentcore_client.create_gateway(
        name=GATEWAY_NAME,
        description="Centralized MCP Gateway demo with Okta OIDC auth",
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": OKTA_DISCOVERY_URL,
                "allowedAudience": ["api://default"],
                "allowedClients": [OKTA_CLIENT_ID],
            }
        },
        exceptionLevel="DEBUG",
    )
    GATEWAY_ID = gateway_response["gatewayId"]
    GATEWAY_URL = gateway_response.get("gatewayUrl", "")
    print(f"Created Gateway: {GATEWAY_ID}")
except agentcore_client.exceptions.ConflictException:
    # Gateway already exists — retrieve it
    gateways = agentcore_client.list_gateways()
    for gw in gateways.get("items", []):
        if gw["name"] == GATEWAY_NAME:
            GATEWAY_ID = gw["gatewayId"]
            break

# Wait for Gateway to be READY
print("Waiting for Gateway to be READY...")
for attempt in range(30):
    gw = agentcore_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = gw["status"]
    if status == "READY":
        GATEWAY_URL = gw["gatewayUrl"]
        print(f"Gateway READY: {GATEWAY_URL}")
        break
    elif status == "FAILED":
        print(f"Gateway FAILED: {gw}")
        break
    time.sleep(5)
    print(f"  Status: {status} ({(attempt + 1) * 5}s)")
else:
    print("Warning: Gateway did not become READY in time")

# --- Credential provider config (Gateway IAM role invokes Lambda) ---
LAMBDA_CRED_CONFIG = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

# --- Step 2: Add Weather Lambda as a target (unrestricted) ---
print("\nAdding targets to Gateway...")

target_ids = {}

try:
    resp = agentcore_client.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name="WeatherTools",
        description="Weather data tools — accessible by all users",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": WEATHER_LAMBDA_ARN,
                    "toolSchema": {
                        "inlinePayload": [{
                            "name": "get_weather",
                            "description": "Get current weather for a city. Available cities: Sydney, New York, London, Tokyo, Singapore.",
                            "inputSchema": {
                                "type": "object",
                                "properties": {
                                    "city": {"type": "string", "description": "City name"}
                                },
                                "required": ["city"]
                            }
                        }]
                    }
                }
            }
        },
        credentialProviderConfigurations=LAMBDA_CRED_CONFIG,
    )
    target_ids["WeatherTools"] = resp["targetId"]
    print(f"  Added target: WeatherTools (Lambda) [{resp['targetId']}]")
except agentcore_client.exceptions.ConflictException:
    print("  WeatherTools target already exists")

# --- Step 3: Add Finance Lambda as a target (restricted) ---
try:
    resp = agentcore_client.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name="FinanceTools",
        description="Financial data tools — RESTRICTED to finance-admins group",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": FINANCE_LAMBDA_ARN,
                    "toolSchema": {
                        "inlinePayload": [{
                            "name": "get_revenue_data",
                            "description": "Get quarterly revenue data. Contains CONFIDENTIAL financial information.",
                            "inputSchema": {
                                "type": "object",
                                "properties": {
                                    "query": {"type": "string", "description": "Revenue query, e.g., 'quarterly revenue' or 'Q1 2026'"}
                                },
                                "required": ["query"]
                            }
                        }]
                    }
                }
            }
        },
        credentialProviderConfigurations=LAMBDA_CRED_CONFIG,
    )
    target_ids["FinanceTools"] = resp["targetId"]
    print(f"  Added target: FinanceTools (Lambda) [{resp['targetId']}]")
except agentcore_client.exceptions.ConflictException:
    print("  FinanceTools target already exists")

# --- Step 4: Add CRM MCP Server as a target (requires HTTPS) ---
if MCP_SERVER_URL and MCP_SERVER_URL.startswith("https"):
    try:
        resp = agentcore_client.create_gateway_target(
            gatewayIdentifier=GATEWAY_ID,
            name="CRMServer",
            description="CRM data tools — existing MCP server on ECS",
            targetConfiguration={
                "mcp": {
                    "mcpServer": {
                        "endpoint": f"{MCP_SERVER_URL}/mcp",
                    }
                }
            },
        )
        target_ids["CRMServer"] = resp["targetId"]
        print(f"  Added target: CRMServer (MCP Server on ECS) [{resp['targetId']}]")
    except agentcore_client.exceptions.ConflictException:
        print("  CRMServer target already exists")
else:
    print("  Skipping CRM MCP Server target — Gateway requires HTTPS endpoints")

print(f"\n  Gateway ready: {GATEWAY_URL}")
print(f"  Gateway ID: {GATEWAY_ID}")
print(f"  Targets: {list(target_ids.keys())}")

## Cell 6: Define Cedar Policies (ENFORCE mode)

Cedar policies define **who can access what** through the Gateway:
- `WeatherTools` — accessible by **all authenticated OAuth users**
- `FinanceTools` — accessible **only by users in the `finance-admins` group**

### Cedar Entity Model for AgentCore Gateway

| Entity | Format | Example |
|--------|--------|---------|
| **Principal** | `AgentCore::OAuthUser` | Any authenticated OAuth user |
| **Action** | `AgentCore::Action::"<TargetName>"` | `AgentCore::Action::"WeatherTools"` |
| **Action (tool)** | `AgentCore::Action::"<Target>___<tool>"` | `AgentCore::Action::"WeatherTools___get_weather"` |
| **Resource** | `AgentCore::Gateway::"<gateway-arn>"` | `AgentCore::Gateway::"arn:aws:..."` |

### Tag-Based Access Control

Okta JWT `groups` claim is mapped as a **tag** on the `AgentCore::OAuthUser` principal.
Cedar policies use `principal.hasTag()` and `principal.getTag()` to evaluate group membership:

```cedar
when {
    principal.hasTag("groups")
    && principal.getTag("groups") like "*finance-admins*"
}
```

This approach is more scalable than hardcoding individual user `sub` values — access is
controlled by managing group membership in Okta, not by updating Cedar policies per user.

See [Cedar policy reference](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy-understanding-cedar.html) for full syntax.

In [ ]:
# --- Ensure Gateway role has policy engine permissions ---
iam_client = boto3.client("iam")
import json as json_mod

pe_policy_doc = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": "bedrock-agentcore:*",
        "Resource": "*"
    }]
}

iam_client.put_role_policy(
    RoleName=f"{DEMO_PREFIX}-gateway-role",
    PolicyName="policy-engine-access",
    PolicyDocument=json_mod.dumps(pe_policy_doc),
)
print("Added bedrock-agentcore permissions to Gateway role")

# --- Create Policy Engine for the Gateway ---
print("\nCreating Policy Engine for access control...\n")

POLICY_ENGINE_NAME = "agentcore_mcp_demo_policy_engine"

try:
    pe_response = agentcore_client.create_policy_engine(
        name=POLICY_ENGINE_NAME,
        description="Cedar policy engine for AgentCore Gateway demo (ENFORCE mode)",
    )
    POLICY_ENGINE_ID = pe_response["policyEngineId"]
    print(f"Created Policy Engine: {POLICY_ENGINE_ID}")
except agentcore_client.exceptions.ConflictException:
    # Already exists — find it
    engines = agentcore_client.list_policy_engines()
    for pe in engines.get("policyEngines", engines.get("items", [])):
        if pe["name"] == POLICY_ENGINE_NAME:
            POLICY_ENGINE_ID = pe["policyEngineId"]
            print(f"Policy Engine already exists: {POLICY_ENGINE_ID}")
            break

# Wait for Policy Engine to be ACTIVE
print("Waiting for Policy Engine to be ACTIVE...")
for attempt in range(30):
    pe = agentcore_client.get_policy_engine(policyEngineId=POLICY_ENGINE_ID)
    status = pe["status"]
    if status == "ACTIVE":
        print(f"Policy Engine ACTIVE")
        break
    elif status == "FAILED":
        print(f"Policy Engine FAILED: {pe}")
        break
    time.sleep(5)
    print(f"  Status: {status} ({(attempt + 1) * 5}s)")

# --- Attach Policy Engine to Gateway in ENFORCE mode ---
print("\nAttaching Policy Engine to Gateway (ENFORCE mode)...")
time.sleep(10)  # Allow IAM propagation
try:
    agentcore_client.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": OKTA_DISCOVERY_URL,
                "allowedAudience": ["api://default"],
                "allowedClients": [OKTA_CLIENT_ID],
            }
        },
        policyEngineConfiguration={
            "arn": pe["policyEngineArn"],
            "mode": "ENFORCE",
        },
        exceptionLevel="DEBUG",
    )
    print("Policy Engine attached to Gateway (ENFORCE mode)")
except Exception as e:
    print(f"Warning: Could not attach policy engine: {e}")

# Wait for Gateway to be READY again after update
print("Waiting for Gateway to settle after policy engine attachment...")
for attempt in range(30):
    gw = agentcore_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
    if gw["status"] == "READY":
        print("Gateway READY")
        break
    time.sleep(5)
    print(f"  Status: {gw['status']} ({(attempt + 1) * 5}s)")

# --- Define Cedar policies ---
# Entity model:
#   Principal: AgentCore::OAuthUser (with tags from JWT claims)
#   Action:    AgentCore::Action::"<TargetName>" or AgentCore::Action::"<Target>___<tool>"
#   Resource:  AgentCore::Gateway::"<gateway-arn>"
# JWT "groups" claim is mapped as a tag on the principal.
# Use principal.hasTag("groups") / principal.getTag("groups") for group-based access.
GATEWAY_ARN = gw["gatewayArn"]

# Policy 1: All authenticated OAuthUsers can access WeatherTools
WEATHER_POLICY = f"""permit(
    principal is AgentCore::OAuthUser,
    action in [AgentCore::Action::"WeatherTools", AgentCore::Action::"WeatherTools___get_weather"],
    resource == AgentCore::Gateway::"{GATEWAY_ARN}"
);"""

# Policy 2: Only users in the "finance-admins" group can access FinanceTools
# The Okta "groups" JWT claim is mapped as a tag on the AgentCore::OAuthUser principal.
# We use principal.hasTag / principal.getTag to check group membership.
# Ref: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy-understanding-cedar.html
FINANCE_POLICY = f"""permit(
    principal is AgentCore::OAuthUser,
    action in [AgentCore::Action::"FinanceTools", AgentCore::Action::"FinanceTools___get_revenue_data"],
    resource == AgentCore::Gateway::"{GATEWAY_ARN}"
) when {{
    principal.hasTag("groups")
    && principal.getTag("groups") like "*finance-admins*"
}};"""

policies = {
    "allow_weather_all": WEATHER_POLICY,
    "allow_finance_group": FINANCE_POLICY,
}

# Create policies on the Policy Engine
for policy_name, policy_body in policies.items():
    try:
        resp = agentcore_client.create_policy(
            name=policy_name,
            policyEngineId=POLICY_ENGINE_ID,
            definition={
                "cedar": {
                    "statement": policy_body.strip()
                }
            },
        )
        print(f"  Policy '{policy_name}' created")
    except agentcore_client.exceptions.ConflictException:
        print(f"  Policy '{policy_name}' already exists")
    except Exception as e:
        print(f"  Warning: Policy '{policy_name}': {e}")

# Verify policies are ACTIVE
time.sleep(10)
print("\nVerifying policy statuses...")
all_active = True
policy_list = agentcore_client.list_policies(policyEngineId=POLICY_ENGINE_ID)
for p in policy_list.get("policies", policy_list.get("items", [])):
    status_icon = "✅" if p["status"] == "ACTIVE" else "❌"
    print(f"  {status_icon} {p['name']}: {p['status']}")
    if p.get("statusReasons"):
        for r in p["statusReasons"]:
            print(f"      {r}")
    if p["status"] != "ACTIVE":
        all_active = False

print("\n--- Policy Summary ---")
print(f"WeatherTools:  ALL authenticated users           -> ALLOW")
print(f"FinanceTools:  users with groups tag matching")
print(f"               'finance-admins'                  -> ALLOW")
print(f"FinanceTools:  all others                        -> DENY (no matching permit)")
print(f"\nMode: ENFORCE (policies enforced by Gateway)")
print(f"Access control: group-based via JWT 'groups' claim mapped as principal tags")
if all_active:
    print("✅ All policies ACTIVE — Gateway-level access control is working")
else:
    print("⚠️  Some policies failed — check status reasons above")

## Cell 7: Save Configuration

Saves all resource identifiers to `gateway_config.json` for the demo notebook.

In [ ]:
config = {
    "gateway_id": GATEWAY_ID,
    "gateway_url": GATEWAY_URL,
    "okta_domain": OKTA_DOMAIN,
    "okta_client_id": OKTA_CLIENT_ID,
    "okta_issuer": OKTA_ISSUER,
    "aws_region": AWS_REGION,
    "policy_engine_id": POLICY_ENGINE_ID,
    "targets": {
        "weather": {"name": "WeatherTools", "type": "lambda", "arn": WEATHER_LAMBDA_ARN,
                     "target_id": target_ids.get("WeatherTools", "")},
        "finance": {"name": "FinanceTools", "type": "lambda", "arn": FINANCE_LAMBDA_ARN,
                     "target_id": target_ids.get("FinanceTools", "")},
        "crm": {"name": "CRMServer", "type": "ecs_mcp_server", "url": MCP_SERVER_URL,
                 "target_id": target_ids.get("CRMServer", "")},
    },
    "demo_resources": {
        "lambda_role": LAMBDA_ROLE_NAME,
        "gateway_role": GATEWAY_ROLE_ARN,
        "weather_lambda": WEATHER_FUNCTION_NAME,
        "finance_lambda": FINANCE_FUNCTION_NAME,
        "ecs_cluster": ECS_CLUSTER_NAME,
        "ecs_service": ECS_SERVICE_NAME,
        "ecr_repo": ECR_REPO_NAME,
        "security_group": sg_id,
    }
}

config_path = os.path.join(os.getcwd(), "gateway_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved to: {config_path}")
print(f"\nConfiguration:")
print(json.dumps(config, indent=2))
print(f"\n🎉 Setup complete! Now run 02_demo.ipynb to see the demo.")

## Cleanup (run after demo is complete)

Run this cell to delete all AWS resources created by this notebook.

- **Terraform destroy** removes all infrastructure: ECR, ECS, Lambda functions, IAM roles, security groups
- **AgentCore API** deletes the Gateway and its targets (no Terraform provider available)

In [ ]:
# ⚠️  CLEANUP — This deletes all demo resources. Only run when you're done!

import json
import subprocess

with open("gateway_config.json") as f:
    cfg = json.load(f)

region = cfg["aws_region"]
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)

gw_id = cfg["gateway_id"]

print("Cleaning up demo resources...\n")

# 1. Delete Gateway targets
for target_key, target_info in cfg.get("targets", {}).items():
    tid = target_info.get("target_id", "")
    if tid:
        try:
            agentcore.delete_gateway_target(gatewayIdentifier=gw_id, targetId=tid)
            print(f"  Deleted target: {target_info['name']} ({tid})")
        except Exception as e:
            print(f"  Skip target {target_info['name']}: {e}")

# 2. Delete Gateway
try:
    agentcore.delete_gateway(gatewayIdentifier=gw_id)
    print(f"  Deleted Gateway: {gw_id}")
except Exception as e:
    print(f"  Skip Gateway: {e}")

# 3. Delete Policy Engine
pe_id = cfg.get("policy_engine_id", "")
if pe_id:
    # Delete policies first
    try:
        policies = agentcore.list_policy_engines()
        # Delete individual policies from the engine
        # (policies are deleted when engine is deleted)
        agentcore.delete_policy_engine(policyEngineId=pe_id)
        print(f"  Deleted Policy Engine: {pe_id}")
    except Exception as e:
        print(f"  Skip Policy Engine: {e}")

# 4. Destroy all infrastructure via Terraform (ECR, ECS, Lambda, IAM, SG)
TERRAFORM_DIR = os.path.join(os.getcwd(), "terraform")
DEMO_PREFIX = "agentcore-mcp-demo"
print("\n  Running terraform destroy...")
try:
    subprocess.run(
        ["terraform", "destroy", "-auto-approve",
         f"-var=aws_region={region}", f"-var=demo_prefix={DEMO_PREFIX}",
         "-var=container_image=placeholder"],
        cwd=TERRAFORM_DIR, check=True,
    )
    print("  ✅ Terraform destroy complete")
except Exception as e:
    print(f"  ⚠️  Terraform destroy failed: {e}")

print("\n✅ Cleanup complete!")